# 7. Reverse Integer

**Difficulty:** Medium  
**Topics:** Math  
**Language:** Python 3  
**Problem:** [LeetCode — Reverse Integer](https://leetcode.com/problems/reverse-integer/)

Reverse an integer's decimal digits while preserving its sign, returning zero if the reversed value is outside the signed 32-bit range. First use a string comparison to check the result before parsing it, then reverse digits arithmetically with overflow checks before every append. Both approaches include a short walkthrough, correctness, complexity, and complete LeetCode-compatible code.

Run the notebook from top to bottom; only Python's standard library is needed. Edit the JSON integer in **Try your own input** to experiment. Copy the final code cell for submission.

## 1. Problem

### Statement summary

Given a signed 32-bit integer `x`, return the integer obtained by reversing its decimal digits and keeping its original sign. If the mathematical reversed result would fall outside `[-2147483648, 2147483647]`, return `0`.

The environment does not permit storing signed or unsigned 64-bit integers. The solution must therefore detect overflow before creating an out-of-range integer, rather than relying on a larger temporary value.

### Constraints

- `-2147483648 <= x <= 2147483647` (equivalently `-2**31 <= x <= 2**31 - 1`).
- No wider integer storage is available for the reversal.

### Examples

**Example 1**

```text
Input: x = 123
Output: 321
```

Reversing the three digits produces `321`.

**Example 2**

```text
Input: x = -123
Output: -321
```

Reverse the digits and retain the negative sign.

**Example 3**

```text
Input: x = 120
Output: 21
```

The reversed digit sequence begins with zero; an integer representation drops that leading zero.

## 2. Understand the task

Return an **integer**, not a string. Preserve a negative sign; it is not a digit to move. Zero reverses to zero, and trailing zeros in the input become insignificant leading zeros in the reversal.

The limits are asymmetric: the negative side allows magnitude `2147483648`, while the positive side stops at `2147483647`. In particular, `abs(-2147483648)` cannot fit in a signed 32-bit integer. We avoid taking that absolute value.

Python integers can grow beyond 32 bits, but using that feature to build an overflowing reversal and check it afterward would miss the problem's storage requirement. The string approach checks decimal text before conversion. The final arithmetic approach keeps every integer intermediate within the signed 32-bit range.

Use supplementary input `x = -120` for both walkthroughs. It combines a negative sign and a trailing zero in just three digits; the answer is `-21`.

In [1]:
MIN_INT = -2147483648
MAX_INT = 2147483647
EXAMPLE_X = -120
EXAMPLE_EXPECTED = -21

## 3. Approach 1 — reverse text and check the bound before parsing

### Idea

Work with decimal characters instead of a numeric magnitude. Remove the sign, reverse the digits, and strip leading zeros. Compare the resulting digit text with the appropriate limit text before converting anything back to an integer.

Equal-length decimal strings compare in numeric order lexicographically. A shorter normalized digit string represents a smaller magnitude. The negative limit is stored as **text**, so no positive integer `2147483648` is created.

### Algorithm

1. Convert the valid input to text and remember whether it begins with `-`.
2. Separate the digits, reverse them, and remove leading zeros. Use `"0"` if no digit remains.
3. Choose the bound text `"2147483648"` for negative input or `"2147483647"` otherwise.
4. Return `0` if the normalized reversal is longer than the bound or, at equal length, lexicographically greater.
5. Restore the sign in the text, then parse that already-validated signed text directly as an integer.

Do not parse a negative result's positive magnitude and negate it afterward; direct signed parsing avoids an unnecessary potentially out-of-range positive intermediate.

### Dry run

`x = -120`

| Stage | State | Explanation |
|---|---|---|
| Separate sign | negative; digits `"120"` | Keep the minus sign out of the reversal |
| Reverse and normalize | `"021"` → `"21"` | Leading zeros do not affect an integer |
| Check magnitude | `"21"` has fewer digits than `"2147483648"` | Reversed result fits |
| Restore and parse | `int("-21")` → `-21` | Return the signed integer |

For supplementary overflow input `1534236469`, the normalized reversal `"9646324351"` is lexicographically greater than `"2147483647"` at equal length. Return `0` before parsing it.

### Why it works

Removing only the sign leaves exactly the input's decimal digits. Reversing them performs the required digit reversal, and deleting leading zeros preserves its integer value. Length comparison, followed by lexicographic comparison at equal length, correctly compares two normalized nonnegative decimal representations. Selecting the sign-specific bound therefore accepts precisely the reversed values that fit. Parsing occurs only after that check, with the sign already restored, so the result is representable without a wider integer intermediate.

### Complexity

Let `d` be the number of decimal digits in the input (take `d = 1` for zero).

- **Worst-case time:** O(d) for text conversion, reversal, normalization, comparison, and bounded parsing.
- **Auxiliary space:** O(d), for digit and sign strings.

Here `d <= 10`, so both are bounded constants under the literal 32-bit input restriction. The digit-based notation explains the work and distinguishes this method's allocations from the arithmetic method.

### Implementation

In [2]:
class Solution:
    def reverse(self, x: int) -> int:
        text = str(x)
        negative = text.startswith("-")
        digits = text[1:] if negative else text
        reversed_digits = digits[::-1].lstrip("0") or "0"
        limit = "2147483648" if negative else "2147483647"
        if len(reversed_digits) > len(limit):
            return 0
        if len(reversed_digits) == len(limit) and reversed_digits > limit:
            return 0
        signed_text = "-" + reversed_digits if negative else reversed_digits
        return int(signed_text)


StringSolution = Solution
print(StringSolution().reverse(EXAMPLE_X))

-21


In [3]:
# Teaching trace, separate from submission code.
def trace_string(x):
    text = str(x)
    negative = text.startswith("-")
    digits = text[1:] if negative else text
    reversed_digits = digits[::-1].lstrip("0") or "0"
    limit = "2147483648" if negative else "2147483647"
    print(f"Input={text}; negative={negative}; digits={digits!r}")
    print(f"Reversed text={digits[::-1]!r}; normalized={reversed_digits!r}; limit={limit!r}")
    if len(reversed_digits) > len(limit):
        print("Too many digits: return 0 before parsing.")
        return 0
    if len(reversed_digits) == len(limit) and reversed_digits > limit:
        print("Magnitude exceeds the bound: return 0 before parsing.")
        return 0
    signed_text = "-" + reversed_digits if negative else reversed_digits
    result = int(signed_text)
    print(f"Parse {signed_text!r}; return {result}")
    return result

assert trace_string(EXAMPLE_X) == EXAMPLE_EXPECTED

Input=-120; negative=True; digits='120'
Reversed text='021'; normalized='21'; limit='2147483648'
Parse '-21'; return -21


## 4. Approach 2 — arithmetic reversal with a negative accumulator

### From temporary strings to safe digit extraction

Replace digit strings with integer division and a running reversed value. Keep both the unprocessed value and accumulator **nonpositive**, including when the original input is positive. Every positive 32-bit input can safely be negated, and an already-negative input needs no negation. This handles the minimum integer without an overflowing absolute value.

Each extracted `digit` is in `[-9, 0]`. Append it with `result * 10 + digit`, but check whether that operation is safe first.

**Python division detail:** `//` rounds down, while removing a signed decimal digit needs truncation toward zero. For example, `-12 // 10` is `-2`, but the desired quotient is `-1`. Compute floor division, then add `1` when `remaining % 10` is nonzero. Set `digit = remaining - quotient * 10`; for `-12`, this gives `-2`. No float conversion or absolute value is needed.

**Check before multiplying.** For either sign, the smallest acceptable prefix before the last append is `-214748364`. If the accumulator equals that prefix, the final negative digit may be no smaller than `-8` for an originally negative input or `-7` for an originally positive input:

```text
result < -214748364                      → overflow
result == -214748364 and digit < limit  → overflow
limit = -8 for negative input, -7 otherwise
```

These checks account for the original sign before the final sign restoration. A positive result cannot use the negative limit's extra unit of magnitude.

### Algorithm

1. Remember the sign. Set `remaining = x` for negative input, otherwise `remaining = -x`.
2. Initialize `result = 0`, cutoff `-214748364`, and the sign-specific last allowed digit.
3. While `remaining` is nonzero, find its quotient truncated toward zero and its final nonpositive decimal digit. Replace `remaining` with the quotient.
4. Reject overflow using the prefix and last-digit checks above.
5. Otherwise append the digit with `result = result * 10 + digit`.
6. Return `result` for negative input or `-result` otherwise. Zero skips the loop and returns zero.

### Dry run

`x = -120`, last allowed digit `-8`

| Visit | Remaining before | Quotient toward zero | Extracted digit | Result before | Guard | Result after |
|---:|---:|---:|---:|---:|---|---:|
| 1 | -120 | -12 | 0 | 0 | safe | 0 |
| 2 | -12 | -1 | -2 | 0 | safe | -2 |
| 3 | -1 | 0 | -1 | -2 | safe | -21 |

The remaining value is now zero, so stop after three visits and return `-21`.

For `1534236469`, the accumulator reaches `-964632435` after nine visits. On the tenth visit, it is already below `-214748364`, so return `0` **without** evaluating the overflowing append.

### Why it works

**Digits are reversed correctly.** Truncating division removes the final decimal digit. The identity `remaining_before = quotient * 10 + digit` extracts that digit with a nonpositive sign. Before each append, `result` represents the already-extracted digits in reverse order, also nonpositive. Multiplying by ten and adding the next digit preserves this property. Leading zero digits naturally disappear.

**The guard is exact.** If `result` is above the cutoff, even the smallest possible next digit `-9` leaves the new value above both permitted lower limits. If it equals the cutoff, comparing the digit with `-8` or `-7` is exactly the remaining boundary test. If it is below the cutoff, appending any digit would be too negative. Further digits cannot shrink the magnitude, so rejecting early is correct.

**Integer intermediates stay representable.** Negating a nonnegative input is safe; the minimum input is never negated. The truncated quotient and `quotient * 10` lie within range. After a successful guard, `result * 10` and the subsequent addition also fit. Finally, a positive-input accumulator is bounded below by `-2147483647`, so negating it is safe. The negative-input result may use the full negative range directly. No wider integer is required.

### Complexity

- **Worst-case time:** O(d), with one constant-work iteration per decimal digit until all digits are processed or overflow is detected.
- **Auxiliary space:** O(1), using only fixed-size numeric and Boolean state. No recursion, strings, or digit lists are needed.

With the official 32-bit restriction there are at most ten iterations. Python's arbitrary precision is not used to store any out-of-range intermediate in this implementation.

### Implementation

In [4]:
class Solution:
    def reverse(self, x: int) -> int:
        negative = x < 0
        remaining = x if negative else -x
        result = 0
        cutoff = -214748364
        last_allowed = -8 if negative else -7

        while remaining != 0:
            quotient = remaining // 10
            if remaining % 10 != 0:
                quotient += 1
            digit = remaining - quotient * 10
            remaining = quotient

            if result < cutoff or (result == cutoff and digit < last_allowed):
                return 0
            result = result * 10 + digit

        return result if negative else -result


ArithmeticSolution = Solution
print(ArithmeticSolution().reverse(EXAMPLE_X))

-21


In [5]:
def trace_arithmetic(x):
    negative = x < 0
    remaining = x if negative else -x
    result = 0
    cutoff = -214748364
    last_allowed = -8 if negative else -7
    visits = 0
    while remaining != 0:
        before = remaining
        quotient = remaining // 10
        if remaining % 10 != 0:
            quotient += 1
        digit = remaining - quotient * 10
        remaining = quotient
        visits += 1
        print(f"Visit {visits}: remaining={before}, quotient={quotient}, digit={digit}, result_before={result}")
        if result < cutoff or (result == cutoff and digit < last_allowed):
            print("  Overflow guard: return 0 before multiplying.")
            return 0
        result = result * 10 + digit
        print(f"  Safe append: result_after={result}")
    answer = result if negative else -result
    print(f"Return {answer}; digit visits={visits}")
    return answer

assert trace_arithmetic(EXAMPLE_X) == EXAMPLE_EXPECTED

Visit 1: remaining=-120, quotient=-12, digit=0, result_before=0
  Safe append: result_after=0
Visit 2: remaining=-12, quotient=-1, digit=-2, result_before=0
  Safe append: result_after=-2
Visit 3: remaining=-1, quotient=0, digit=-1, result_before=-2
  Safe append: result_after=-21
Return -21; digit visits=3


## 5. Comparison and common mistakes

| Approach | Time in decimal digits | Auxiliary space | Main trade-off |
|---|---|---|---|
| Reverse text, compare bounds, then parse | O(d) | O(d) | Direct representation; allocates strings |
| Guarded arithmetic with negative state | O(d) | O(1) | Avoids digit storage; requires careful signed division |

Choose the guarded arithmetic method for submission. Both approaches check representability before creating the result; the arithmetic implementation makes every intermediate bound explicit. Under the fixed 32-bit domain, `d <= 10` bounds both methods' total work.

Common mistakes:

- Computing an unrestricted reversal first and only then checking bounds.
- Taking `abs(-2147483648)` in a fixed-width implementation.
- Reversing the minus sign as if it were a digit.
- Using Python floor division on a negative value without adjustment toward zero.
- Assuming `% 10` directly yields a negative last digit in Python.
- Applying the positive upper bound symmetrically without reasoning about the negative minimum.
- Returning a string or preserving insignificant leading zeros.
- Comparing digit strings lexicographically before normalizing zeros or checking their lengths.

The platform algorithms assume valid signed 32-bit integer input. The manual harness separately rejects invalid types and out-of-range values. Python may represent larger numbers in that validation or in test oracles; this is testing infrastructure, not part of the submission algorithm.

## 6. Test the solutions

The first three cases are all official examples. Supplementary cases cover zero, a negative trailing zero, single digits, multiple trailing zeros, both input limits, reversals near the boundary, and overflow with either sign. Check every implementation against exact integer results.

In [6]:
TEST_CASES = [
    ("Problem example 1", 123, 321),
    ("Problem example 2", -123, -321),
    ("Problem example 3", 120, 21),
    ("Supplementary: zero", 0, 0),
    ("Supplementary: negative trailing zero", -120, -21),
    ("Supplementary: single digit", -7, -7),
    ("Supplementary: multiple zeros", 1000, 1),
    ("Supplementary: positive overflow", 1534236469, 0),
    ("Supplementary: negative overflow", -1534236469, 0),
    ("Supplementary: maximum input", 2147483647, 0),
    ("Supplementary: minimum input", -2147483648, 0),
    ("Supplementary: near positive boundary", 1463847412, 2147483641),
    ("Supplementary: near negative boundary", -1463847412, -2147483641),
    ("Supplementary: just beyond reversed prefix", 1563847412, 0),
    ("Supplementary: large valid reversal", 1000000002, 2000000001),
]
for name, x, expected in TEST_CASES:
    for implementation in (StringSolution, ArithmeticSolution):
        result = implementation().reverse(x)
        assert type(result) is int and result == expected, (name, result, expected)
        assert MIN_INT <= result <= MAX_INT
    print(f"PASS | {name}: {expected}")

PASS | Problem example 1: 321
PASS | Problem example 2: -321
PASS | Problem example 3: 21
PASS | Supplementary: zero: 0
PASS | Supplementary: negative trailing zero: -21
PASS | Supplementary: single digit: -7
PASS | Supplementary: multiple zeros: 1
PASS | Supplementary: positive overflow: 0
PASS | Supplementary: negative overflow: 0
PASS | Supplementary: maximum input: 0
PASS | Supplementary: minimum input: 0
PASS | Supplementary: near positive boundary: 2147483641
PASS | Supplementary: near negative boundary: -2147483641
PASS | Supplementary: just beyond reversed prefix: 0
PASS | Supplementary: large valid reversal: 2000000001


## 7. Try your own input

Edit `x_text` and optionally `approach`, then run the next cell again. Use a JSON integer such as `"-120"`, without extra quotes around the number inside the JSON. Choose `"arithmetic"` or `"string"`.

The harness rejects booleans, decimal values, strings, and integers outside the signed 32-bit input range. A valid input whose reversal overflows returns `0`; it is not a harness error. Parsing and validation belong to the notebook interface, separately from the submission code. Run All never waits for interactive input.

In [7]:
import json

def run_manual(x_text, approach="arithmetic"):
    try:
        x = json.loads(x_text)
        if type(x) is not int or not MIN_INT <= x <= MAX_INT:
            raise ValueError("x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).")
        implementations = {"string": StringSolution, "arithmetic": ArithmeticSolution}
        if approach not in implementations:
            raise ValueError("approach must be 'string' or 'arithmetic'.")
        result = implementations[approach]().reverse(x)
        print(f"{approach}: x={x}; reversed result={result}")
        return result
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then run this cell again.
x_text = "-120"
approach = "arithmetic"  # Or "string".
manual_result = run_manual(x_text, approach)

arithmetic: x=-120; reversed result=-21


In [8]:
assert run_manual("120", "string") == 21
assert run_manual("1534236469") == 0
assert run_manual("-2147483648") == 0
assert run_manual("true") is None
assert run_manual("123.0") is None
assert run_manual('"123"') is None
assert run_manual("[123]") is None
assert run_manual("2147483648") is None
assert run_manual("-2147483649") is None
assert run_manual("not JSON") is None
assert run_manual("123", "unknown") is None
print("Manual-input harness checks passed.")

string: x=120; reversed result=21
arithmetic: x=1534236469; reversed result=0
arithmetic: x=-2147483648; reversed result=0
Input error: x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).
Input error: x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).
Input error: x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).
Input error: x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).
Input error: x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).
Input error: x must be a signed 32-bit JSON integer (-2147483648 to 2147483647).
Input error: Expecting value: line 1 column 1 (char 0)
Input error: approach must be 'string' or 'arithmetic'.
Manual-input harness checks passed.


## 8. Final submission

Copy the next cell into LeetCode's Python 3 editor. It uses the verified `Solution.reverse(self, x: int) -> int` interface and guarded arithmetic reversal. It needs no imports or notebook helpers and never constructs an out-of-range integer intermediate.

In [9]:
class Solution:
    def reverse(self, x: int) -> int:
        negative = x < 0
        remaining = x if negative else -x
        result = 0
        cutoff = -214748364
        last_allowed = -8 if negative else -7

        while remaining != 0:
            quotient = remaining // 10
            if remaining % 10 != 0:
                quotient += 1
            digit = remaining - quotient * 10
            remaining = quotient

            if result < cutoff or (result == cutoff and digit < last_allowed):
                return 0
            result = result * 10 + digit

        return result if negative else -result